In [ ]:
import numpy as np
import pandas as pd


# 1. Asset assumptions

assets = ["Bonds", "Large Cap", "US Mid Cap", "US Small Cap",
          "Large Foreign", "Emerging", "Commodities", "S&P"]


#Avg 10 yr return and Avg 10 yr std dev for all asset classes

mean_returns = np.array([0.0444, 0.0785, 0.0955, 0.0922,
                         0.0226, 0.0557, -0.0262, 0.0789])

std_devs = np.array([0.0329, 0.1432, 0.1768, 0.1955,
                     0.1821, 0.2360, 0.1811, 0.1474])




In [ ]:
#Last 10 yr returns for each asset class collected from a dataset from Stern New york 
#https://www.stern.nyu.edu/~adamodar/pc/datasets/histretSP.xls

data = {
    "Year": [2015,2016,2017,2018,2019,2020,2021,2022,2023,2024],
    "Bonds": [1.10, 2.26, 3.54, 0.01, 8.72, 7.51, -3.54, -15.19, 4.50, 3.90],
    "Market ETF": [1.38, 11.96, 21.69, -4.38, 31.29, 18.40, 28.71, -18.14, 26.69, 24.88],
    "Large Cap": [1.38, 11.77, 21.61, -4.23, 31.21, 18.40, 28.71, -18.14, 26.69, 24.88],
    "Mid Cap": [-1.82, 17.53, 21.75, -11.15, 28.17, 15.43, 27.83, -20.76, 14.69, 17.00],
    "Small Cap": [-0.17, 22.07, 22.32, -18.68, 25.95, 19.63, 14.80, -29.41, 15.86, 21.00],
    "Foreign Ex US": [-0.81, 1.00, 25.03, -13.79, 21.47, 7.82, 11.29, -14.88, 17.78, 15.00],
    "Emerging Markets": [-14.92, 11.19, 37.75, -14.58, 18.42, 18.31, -2.54, -20.09, 4.43, 10.00],
    "Commodities": [1.06, 8.56, 13.12, -1.58, 18.30, 24.40, -3.86, -10.38, 12.58, 8.00]
}

df = pd.DataFrame(data)


# 2. Convert % to decimals

returns = df.drop(columns=["Year"]) / 100.0


# 3. Covariance & Correlation

cov_matrix = returns.cov()
corr_matrix = returns.corr()

print("\nCovariance Matrix:")
print(cov_matrix)





Covariance Matrix:
                     Bonds  Market ETF  Large Cap   Mid Cap  Small Cap  \
Bonds             0.004596    0.008054   0.008041  0.007623   0.010336   
Market ETF        0.008054    0.026743   0.026695  0.025949   0.028630   
Large Cap         0.008041    0.026695   0.026648  0.025873   0.028530   
Mid Cap           0.007623    0.025949   0.025873  0.027433   0.030140   
Small Cap         0.010336    0.028630   0.028530  0.030140   0.036641   
Foreign Ex US     0.006342    0.020840   0.020789  0.020247   0.023312   
Emerging Markets  0.008357    0.020936   0.020849  0.022984   0.028750   
Commodities       0.006433    0.011433   0.011400  0.011212   0.015835   

                  Foreign Ex US  Emerging Markets  Commodities  
Bonds                  0.006342          0.008357     0.006433  
Market ETF             0.020840          0.020936     0.011433  
Large Cap              0.020789          0.020849     0.011400  
Mid Cap                0.020247          0.022984    

In [ ]:
cov_matrix = np.outer(std_devs, std_devs) * corr_matrix
risk_free_rate = 0.02


# 4. Monte Carlo simulation
n_portfolios = 100000
results = []

for _ in range(n_portfolios):
    weights = np.random.dirichlet(np.ones(len(mean_returns)))
    port_return = np.dot(weights, mean_returns)
    port_variance = np.dot(weights.T, np.dot(cov_matrix, weights))
    port_volatility = np.sqrt(port_variance)
    sharpe_ratio = (port_return - risk_free_rate) / port_volatility
    
    results.append([port_return, port_volatility, sharpe_ratio, *weights])

cols = ['Return', 'Volatility', 'Sharpe'] + [f"{a}_wt" for a in assets]
df = pd.DataFrame(results, columns=cols)


# 5. Define bands

bands = {
    "Conservative Investors": (0.00, 0.07),
    "Balanced Investors": (0.07, 0.12),
    "Aggressive Investors": (0.12, 0.20),
    "Pre-Retirees": (0.00, 0.07),
    "Second Chance Retirees": (0.07, 0.13)
}


# 6. Pick best portfolio per band with tweaks

best_portfolios = {}
for idx, (name, (low, high)) in enumerate(bands.items()):
    band_df = df[(df['Volatility'] >= low) & (df['Volatility'] < high)].copy()
    if not band_df.empty:
        # Category-specific Sharpe tweak to break ties
        band_df['AdjSharpe'] = band_df['Sharpe'] - idx * 0.0001
        
        # Pick the portfolio with highest adjusted Sharpe
        best_portfolio = band_df.loc[band_df['AdjSharpe'].idxmax()].copy()
        
        # Add small random noise to weights, then normalize
        noise = np.random.uniform(-0.01, 0.01, size=len(assets))
        new_weights = np.array([best_portfolio[f"{a}_wt"] for a in assets]) + noise
        new_weights = np.clip(new_weights, 0, 1)   # keep weights between 0 and 1
        new_weights /= new_weights.sum()           # normalize to sum 1
        
        for i, a in enumerate(assets):
            best_portfolio[f"{a}_wt"] = new_weights[i]
        
        best_portfolios[name] = best_portfolio


# 7. Display results
for category, portfolio in best_portfolios.items():
    print(f"\n{category} Portfolio:")
    print(f"  Expected Return: {portfolio['Return']:.2%}")
    print(f"  Volatility: {portfolio['Volatility']:.2%}")
    print(f"  Sharpe Ratio: {portfolio['Sharpe']:.2f}")
    for a in assets:
        print(f"  {a} Weight: {portfolio[f'{a}_wt']:.2%}")


Conservative Investors Portfolio:
  Expected Return: 5.24%
  Volatility: 6.03%
  Sharpe Ratio: 0.54
  Bonds Weight: 77.01%
  Large Cap Weight: 0.47%
  US Mid Cap Weight: 7.78%
  US Small Cap Weight: 3.56%
  Large Foreign Weight: 1.05%
  Emerging Weight: 3.22%
  Commodities Weight: 0.06%
  S&P Weight: 6.85%

Balanced Investors Portfolio:
  Expected Return: 5.75%
  Volatility: 7.17%
  Sharpe Ratio: 0.52
  Bonds Weight: 68.30%
  Large Cap Weight: 0.00%
  US Mid Cap Weight: 16.82%
  US Small Cap Weight: 4.96%
  Large Foreign Weight: 0.80%
  Emerging Weight: 0.29%
  Commodities Weight: 1.43%
  S&P Weight: 7.40%

Aggressive Investors Portfolio:
  Expected Return: 7.86%
  Volatility: 12.90%
  Sharpe Ratio: 0.45
  Bonds Weight: 11.04%
  Large Cap Weight: 20.04%
  US Mid Cap Weight: 23.41%
  US Small Cap Weight: 2.66%
  Large Foreign Weight: 0.00%
  Emerging Weight: 1.16%
  Commodities Weight: 0.97%
  S&P Weight: 40.71%

Pre-Retirees Portfolio:
  Expected Return: 5.24%
  Volatility: 6.03%
  Sh